In [ ]:
#!pip install -r requirements.txt

^C


In [4]:
import torch
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
print(torch.__version__)

2.11.0+cpu


In [2]:
from reasoning_from_scratch.qwen3 import download_qwen3_small
download_qwen3_small(kind = "base", tokenizer_only = True, out_dir = "qwen3")

In [10]:
tokenizer_file_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_file_path)

In [11]:
demo = "hi my name is sanchit"
input_token = tokenizer.encode(demo)
print(f"input: {input_token}")
for token in input_token:
    print(f"{token} -> {tokenizer.decode([token])}")

input: [6023, 847, 829, 374, 274, 3497, 275]
6023 -> hi
847 ->  my
829 ->  name
374 ->  is
274 ->  s
3497 -> anch
275 -> it


In [6]:
device = torch.device("cpu")

In [17]:
download_qwen3_small(kind = "base", tokenizer_only = False, out_dir = "qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [7]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B
model_file = Path("qwen3")/"qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_file))
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [12]:
out = model(torch.tensor(input_token).unsqueeze(dim=0))
out.shape

torch.Size([1, 7, 151936])

### Demo testing text gen

In [ ]:
@torch.inference_mode()
def generate_text_basic(model, input:torch.tensor, max_new_tokens,
                        end_token:int = None) -> torch.tensor:
    input_length = input.shape[1]
    model.eval()

    for _ in range(max_new_tokens):
        output = model(input)
        logits = output[:, -1, :]
        out_token = torch.argmax(logits, dim=-1, keepdim=True)

        if end_token is not None:
            if torch.all(out_token == end_token): 
                #check for ALL batches => simulatenously must be true else continue generation (good for 1 batch size only)
                break

        input = torch.cat([input, out_token], dim = 1)

    return input[:, input_length:]

In [19]:
prompt = "Explain large language models in a single sentence."
input_token_ids_tensor = torch.tensor(tokenizer.encode(prompt),device=device).unsqueeze(0)

max_new_tokens = 10


for token in generate_text_basic(
    model=model, input=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,end_token = tokenizer.eos_token_id):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True  # Deactivates buffering so tokens are printed live
        )

 Large language models are artificial intelligence systems that can understand

#### Benchmarking

In [20]:
import warnings

def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            # Check whether we are actually using this backend
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
    
            # Synchronize if supported (important for async backends)
            if hasattr(backend, "synchronize"):
                backend.synchronize()
            
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [23]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic(
    model=model,
    input=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    end_token=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand

Time: 6.99 sec
1 tokens/sec


### KV caching 

In [33]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_cache(model, input:torch.tensor, max_new_tokens, end_token:int = None) -> torch.tensor:
    
    input_length = input.shape[1]
    model.eval()
    cache = KVCache(n_layers = model.cfg["n_layers"]) #creates storage for KV caching
    model.reset_kv_cache()         

    out = model(input, cache=cache)[:, -1, :] #first pass entire input                      

    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if end_token is not None and torch.all(next_token == end_token): 
                break

        input = torch.cat([input, next_token], dim = 1) #not really required anymore
        out = model(next_token, cache)[:, -1, :] # only next token is passed not entire sequence!!!

    return input[:, input_length:]

In [34]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_cache(
    model=model,
    input=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    end_token=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand

Time: 2.04 sec
4 tokens/sec


4x Speedup

### Model Compilation

In [ ]:
model_compiled = torch.compile(model, mode = 'max-autotune') 

In [41]:
for i in range(3):
    
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic(
        model=model_compiled,
        input=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        end_token=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}\n")

InductorError: RuntimeError: Compiler: cl is not found.

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


Not working at the moment, will have to install Visual Studio Build Tools with the "C++ workload" and run Python from the "x64 Native Tools" prompt.


TLDR of compiling: speeds up inference by another x2-3 which with KV caching gives an effective ~x8-10 speedup